# 📊 LAB 1: THU THẬP DỮ LIỆU WEB (WEB SCRAPING)\n**Môn học:** DS200 - Phân tích dữ liệu lớn (Big Data Analytics)  \n**Giảng viên hướng dẫn:** ThS. Hà Minh Tân  \n**Chủ đề:** Khai phá & Thu thập dữ liệu từ Website bằng Python (Requests & BeautifulSoup4)\n\n---\n\n### 🚀 Mở và chạy trực tiếp trên Google Colab:\n[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/barackvn/MayHocNangCao/blob/main/Buoi2-KhaiPhaVaXulyDuLieu/Lab1_ThuThapDuLieu_WebScraping.ipynb)\n\n> *Nhấn vào nút trên để mở và chạy trực tiếp notebook trên Google Colab.*

## 📑 Mục tiêu bài Lab
1. **Hiểu cơ chế hoạt động:** Nắm vững giao thức HTTP/HTTPS, cách client gửi request và server trả lời response (Status Code: 200, 403, 404...).
2. **Làm quen công cụ thu thập:** Sử dụng thư viện `requests` để tải HTML và `BeautifulSoup4` để phân tích cây cấu trúc DOM.
3. **Thực hành thu thập:** Crawl danh sách tin tức bóng đá từ VnExpress (`https://vnexpress.net/bong-da`).
4. **Trích xuất thông tin chi tiết:** Bóc tách tiêu đề (`title`), tóm tắt (`abstract`/`description`), thời gian đăng, nội dung từng bài viết.
5. **Yêu cầu 5 (Bắt buộc):** Tổ chức dữ liệu vào Pandas DataFrame và xuất ra file CSV (`data_vnexpress_bongda.csv`).
6. **Yêu cầu 6 (Mở rộng):** Thu thập dữ liệu từ chuyên mục hoặc trang tin tức khác (ví dụ: Dân Trí / VnExpress Thời sự) để so sánh cấu trúc DOM.

--- 
## Bước 1: Cài đặt và nạp các thư viện cần thiết
Trên Google Colab, các thư viện như `requests`, `pandas`, `beautifulsoup4` thường đã có sẵn. Ta cài đặt/cập nhật thêm để đảm bảo môi trường đầy đủ nhất.

In [ ]:
# Cài đặt thư viện theo tài liệu bài lab (chạy dấu ! trên Colab)
!pip install -q requests beautifulsoup4 pillow pandas lxml

In [ ]:
# Nạp các thư viện cần thiết
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from datetime import datetime

print("✅ Các thư viện đã được import thành công!")
print(f"- Requests version: {requests.__version__}")
print(f"- Pandas version: {pd.__version__}")

--- 
## Bước 2: Gửi HTTP Request lấy mã nguồn trang Web
- Thư viện `requests` đóng vai trò như một trình duyệt ảo gửi yêu cầu `GET` tới URL mục tiêu: `https://vnexpress.net/bong-da`.
- Server trả về đối tượng `Response`. Mã trạng thái `200` biểu thị yêu cầu thành công.

In [ ]:
# Thiết lập URL và User-Agent header (giả lập trình duyệt để tránh bị chặn IP/bot)
url = "https://vnexpress.net/bong-da"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# Gửi yêu cầu GET
response = requests.get(url, headers=headers)

# Kiểm tra phản hồi trả về từ máy chủ
print("HTTP Response:", response)
print("Status Code:", response.status_code)
print("Encoding:", response.encoding)

In [ ]:
# Xem thử một đoạn dữ liệu thô (HTML) trả về
print("500 ký tự đầu tiên của trang web:")
print(response.text[:500])

--- 
## Bước 3: Tách dữ liệu HTML bằng BeautifulSoup4
- `BeautifulSoup` sẽ phân tích cú pháp chuỗi HTML thô thành một cấu trúc cây (DOM Tree), giúp chúng ta dễ dàng tìm kiếm và bóc tách các thẻ như `<div>`, `<h2>`, `<a>`, `<p>`...

In [ ]:
# Phân tích HTML bằng html.parser
soup = BeautifulSoup(response.content, "html.parser")

# In thử tiêu đề trang web
print("Tiêu đề trang web (Page Title):", soup.title.string if soup.title else "None")

--- 
## Bước 4: Lấy danh sách liên kết bài viết (Links)
- Trên trang VnExpress, các tiêu đề bài báo thường nằm trong thẻ `<h2>` hoặc `<h3>` có thuộc tính `class="title-news"`.
- Bên trong mỗi tiêu đề sẽ có thẻ `<a>` chứa thuộc tính `href` là đường link chi tiết dẫn tới bài báo.

In [ ]:
# Tìm tất cả các thẻ tiêu đề tin tức
titles = soup.find_all(["h2", "h3"], class_="title-news")
print(f"Tìm thấy {len(titles)} bài viết trên trang danh mục.")

# Bóc tách danh sách URL bài báo
links = []
for item in titles:
    a_tag = item.find("a")
    if a_tag and "href" in a_tag.attrs:
        href = a_tag["href"]
        # Chỉ lấy các bài viết có đuôi .html hợp lệ và bắt đầu bằng http
        if href.startswith("https://vnexpress.net") and href.endswith(".html"):
            if href not in links:
                links.append(href)

print(f"✅ Đã lọc được {len(links)} đường link bài viết duy nhất.")
print("\n--- 5 link đầu tiên:")
for i, link in enumerate(links[:5], start=1):
    print(f"{i}. {link}")

--- 
## Bước 5: Bóc tách dữ liệu chi tiết của từng bài báo
Với mỗi đường link bài báo, chúng ta gửi request vào trang chi tiết và bóc tách:
- **Tiêu đề bài viết:** Thẻ `<h1 class="title-detail">`
- **Mô tả ngắn (Sapo / Abstract):** Thẻ `<p class="description">`
- **Thời gian xuất bản:** Thẻ `<span class="date">`
- **Nội dung bài viết:** Các đoạn thẻ `<p class="Normal">` bên trong `<article class="fck_detail">`

> **Mẹo kỹ thuật:** Thêm khoảng dừng `time.sleep(0.3)` giữa các request để thể hiện hành vi crawl văn minh (Polite Scraping) và tránh bị khóa IP tạm thời.

In [ ]:
# Danh sách lưu trữ thông tin bóc tách
articles_data = []

# Thu thập tối đa 15 bài đầu tiên để demo nhanh (có thể tăng số lượng tùy nhu cầu)
max_articles = 15
target_links = links[:max_articles]

print(f"🚀 Bắt đầu thu thập dữ liệu chi tiết cho {len(target_links)} bài viết...\n")

for idx, link in enumerate(target_links, start=1):
    try:
        news_resp = requests.get(link, headers=headers, timeout=10)
        if news_resp.status_code != 200:
            continue
            
        news_soup = BeautifulSoup(news_resp.content, "html.parser")
        
        # 1. Tiêu đề
        title_tag = news_soup.find("h1", class_="title-detail") or news_soup.find("h1")
        title = title_tag.get_text(strip=True) if title_tag else ""
        
        # 2. Tóm tắt / Mô tả
        desc_tag = news_soup.find("p", class_="description")
        if desc_tag:
            abstract = desc_tag.get_text(strip=True)
        else:
            # Dự phòng bằng meta description
            meta_desc = news_soup.find("meta", {"name": "description"})
            abstract = meta_desc.get("content", "").strip() if meta_desc else ""
            
        # 3. Thời gian đăng
        date_tag = news_soup.find("span", class_="date")
        publish_date = date_tag.get_text(strip=True) if date_tag else ""
        
        # 4. Nội dung bài viết (ghép các đoạn văn bản chính)
        paragraphs = news_soup.select("article.fck_detail p.Normal")
        content = "\n".join([p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)])
        
        # Lưu vào danh sách
        articles_data.append({
            "id": idx,
            "title": title,
            "abstract": abstract,
            "publish_date": publish_date,
            "content": content,
            "url": link,
            "crawled_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })
        
        print(f"[{idx}/{len(target_links)}] Đã lấy: {title[:60]}...")
        
        # Nghỉ ngắn giữa các request
        time.sleep(0.3)
        
    except Exception as e:
        print(f"❌ Lỗi khi tải bài viết {link}: {e}")

print(f"\n🎉 Thu thập hoàn tất! Tổng cộng {len(articles_data)} bài viết.")

--- 
## Bước 6: Yêu cầu 5 - Lưu trữ dữ liệu thành file CSV
- Chuyển dữ liệu sang `pandas.DataFrame` để quản lý dạng bảng chuyên nghiệp.
- Xuất ra file `data_vnexpress_bongda.csv` với định dạng mã hóa `utf-8-sig` (BOM) để khi mở bằng **Microsoft Excel** không bị lỗi font tiếng Việt.

In [ ]:
# Tạo DataFrame từ dữ liệu đã crawl
df_articles = pd.DataFrame(articles_data)

# Hiển thị 5 dòng đầu tiên
print("Mẫu dữ liệu thu thập được:")
df_articles[["id", "title", "publish_date", "abstract"]].head()

In [ ]:
# Kiểm tra thông tin cấu trúc DataFrame
df_articles.info()

In [ ]:
# Lưu dữ liệu ra file CSV
csv_filename = "data_vnexpress_bongda.csv"
df_articles.to_csv(csv_filename, index=False, encoding="utf-8-sig")

print(f"💾 Đã xuất dữ liệu thành công ra file: {csv_filename}")
print(f"- Kích thước tập tin: {len(df_articles)} bản ghi, {df_articles.shape[1]} cột")

In [ ]:
# (Tùy chọn trên Colab) Tải file CSV về máy tính cá nhân
try:
    from google.colab import files
    files.download(csv_filename)
    print("⬇️ Đang gửi file tải về máy của bạn...")
except ImportError:
    print("Đang chạy ở môi trường máy cục bộ (Local), file đã lưu tại thư mục hiện tại.")

--- 
## Bước 7: Yêu cầu 6 - Mở rộng: Thu thập dữ liệu từ Website/Chuyên mục khác
Thực hành áp dụng phương pháp cào dữ liệu cho một trang báo khác: **Báo Dân Trí (`dantri.com.vn`)** hoặc chuyên mục **Thời sự của VnExpress** để so sánh sự khác nhau về cấu trúc HTML.

In [ ]:
# Hàm tổng quát bóc tách dữ liệu báo Dân Trí (Chuyên mục Thể thao)
def crawl_dantri_thethao(limit=10):
    dantri_url = "https://dantri.com.vn/the-thao.htm"
    res = requests.get(dantri_url, headers=headers, timeout=10)
    soup = BeautifulSoup(res.content, "html.parser")
    
    # Thẻ bài viết trên Dân Trí thường nằm trong h3.article-title
    article_tags = soup.select("h3.article-title a")
    dantri_links = []
    for a in article_tags:
        href = a.get("href", "")
        if href.endswith(".htm"):
            full_url = "https://dantri.com.vn" + href if href.startswith("/") else href
            if full_url not in dantri_links:
                dantri_links.append(full_url)
                
    print(f"Tìm thấy {len(dantri_links)} bài báo trên Dân Trí Thể thao. Bắt đầu lấy {limit} bài...")
    
    results = []
    for idx, link in enumerate(dantri_links[:limit], start=1):
        try:
            r = requests.get(link, headers=headers, timeout=10)
            sp = BeautifulSoup(r.content, "html.parser")
            
            title = sp.find("h1").get_text(strip=True) if sp.find("h1") else ""
            desc = sp.find("h2").get_text(strip=True) if sp.find("h2") else ""
            
            results.append({
                "id": idx,
                "source": "Dân Trí",
                "title": title,
                "abstract": desc,
                "url": link,
                "crawled_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            })
            print(f"  [{idx}] {title[:60]}...")
            time.sleep(0.3)
        except Exception as err:
            print(f"  Lỗi bài {link}: {err}")
            
    return pd.DataFrame(results)

# Chạy thử nghiệm bóc tách Dân Trí
df_dantri = crawl_dantri_thethao(limit=5)
df_dantri.to_csv("data_dantri_thethao.csv", index=False, encoding="utf-8-sig")
print("\n✅ Đã lưu kết quả crawl Dân Trí vào 'data_dantri_thethao.csv'")
df_dantri.head()

--- 
## 🎯 Bảng so sánh cấu trúc DOM giữa các trang Web
| Yếu tố dữ liệu | VnExpress (`vnexpress.net`) | Dân Trí (`dantri.com.vn`) |
| :--- | :--- | :--- |
| **Khung chứa danh sách** | `h2.title-news`, `h3.title-news` | `h3.article-title` |
| **Đường dẫn bài viết** | Thẻ `<a>` trong tiêu đề (`href`) | Thẻ `<a>` trong tiêu đề (`href`) |
| **Tiêu đề chi tiết** | `h1.title-detail` hoặc `h1` | `h1` (Tailwind classes) |
| **Mô tả ngắn (Sapo)** | `p.description` | `h2` dưới tiêu đề |
| **Nội dung bài viết** | `article.fck_detail p.Normal` | `div.singular-content p` |

--- 
## 💡 Kết luận và Bài học kinh nghiệm
1. **Tính biến động của DOM:** Cấu trúc HTML của các trang web có thể thay đổi hoặc có các định dạng bài báo khác nhau (video, e-magazine, phóng sự ảnh). Do đó, khi lập trình Web Scraping cần có các cơ chế xử lý ngoại lệ (`try...except`) và fallback (dự phòng) dữ liệu.
2. **Quy tắc văn minh (Robots.txt & Rate Limit):** Khi cào dữ liệu lớn, cần tuân thủ tệp `robots.txt` của website và thiết lập độ trễ giữa các request (`time.sleep`) để không làm quá tải máy chủ máy đích.
3. **Lưu trữ chuẩn tiếng Việt:** Khi lưu trữ dữ liệu dạng văn bản tiếng Việt sang CSV, luôn sử dụng `encoding='utf-8-sig'` để tránh lỗi vỡ font khi mở trên MS Excel.